# Projeto de Integração de Dados de Criminalidade e Meteorologia em Passo Fundo

## Etapa 1 – Coleta e Organização dos Dados

Nesta etapa, foi realizada a organização inicial dos datasets necessários para o projeto, separando os dados de **criminalidade** e **meteorologia** em uma estrutura padronizada no Google Drive.

O objetivo desta etapa é garantir que os dados estejam organizados, identificáveis e acessíveis para todos os integrantes do grupo no Google Colab.

In [ ]:
import pandas as pd

In [ ]:
#Conectar o Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Definir a pasta principal do projeto
import os

base_path = "/content/drive/MyDrive/projeto_criminalidade_passo_fundo"

In [ ]:
#Criar automaticamente a estrutura de pastas
pastas = [
    "data/raw/data_criminalidade",
    "data/raw/data_meteorologia",
    "data/interim",
    "data/processed",
    "notebooks",
    "outputs/graficos",
    "outputs/tabelas",
    "docs"
]

for pasta in pastas:
    os.makedirs(os.path.join(base_path, pasta), exist_ok=True)

print("Estrutura de pastas criada com sucesso!")

In [ ]:
#Verificar se as pastas foram criadas
for raiz, dirs, files in os.walk(base_path):
    nivel = raiz.replace(base_path, "").count(os.sep)
    indent = " " * 4 * nivel
    print(f"{indent}{os.path.basename(raiz)}/")
    subindent = " " * 4 * (nivel + 1)
    for file in files:
        print(f"{subindent}{file}")

## Organização dos dados

Os arquivos foram organizados nas seguintes categorias:

- **Criminalidade:** base principal de ocorrências criminais no RS
- **Meteorologia:** arquivos meteorológicos por localidade, incluindo Passo Fundo

### Arquivos principais identificados

- **Criminalidade:** `criminalidade_ocorrencias_rs_2026.csv`
- **Meteorologia (principal):** `PASSO_FUNDO_2021-01-01_2025-09-05.csv`

Além disso, foram mantidos arquivos meteorológicos de cidades próximas para eventual uso complementar ou comparação.

In [ ]:
#Definir caminhos dos dados
crime_path = os.path.join(base_path, "data/raw/data_criminalidade")
meteo_path = os.path.join(base_path, "data/raw/data_meteorologia")

In [ ]:
#Listar arquivos disponíveis
arquivos_crime = os.listdir(crime_path)
arquivos_meteo = os.listdir(meteo_path)

print("Arquivos de criminalidade:")
for arq in arquivos_crime:
    print("-", arq)

print("\nArquivos meteorológicos:")
for arq in arquivos_meteo:
    print("-", arq)

In [ ]:
#Criar inventário dos arquivos
def listar_arquivos_com_tamanho(caminho, tipo):
    arquivos = []
    for arq in os.listdir(caminho):
        caminho_completo = os.path.join(caminho, arq)
        if os.path.isfile(caminho_completo):
            tamanho_mb = os.path.getsize(caminho_completo) / (1024 * 1024)
            arquivos.append({
                "tipo": tipo,
                "arquivo": arq,
                "tamanho_mb": round(tamanho_mb, 2)
            })
    return arquivos

lista_crime = listar_arquivos_com_tamanho(crime_path, "criminalidade")
lista_meteo = listar_arquivos_com_tamanho(meteo_path, "meteorologia")

df_arquivos = pd.DataFrame(lista_crime + lista_meteo)
df_arquivos

In [ ]:
#Resumo da coleta
print("Resumo da coleta de dados")
print("-" * 40)
print(f"Total de arquivos de criminalidade: {len(arquivos_crime)}")
print(f"Total de arquivos meteorológicos: {len(arquivos_meteo)}")
print(f"Total geral: {len(arquivos_crime) + len(arquivos_meteo)}")

In [ ]:
#Inspeção inicial da base criminal
arquivo_crime = os.path.join(crime_path, "criminalidade_ocorrencias_rs_2026.csv")

try:
    df_crime_preview = pd.read_csv(arquivo_crime, sep=";", encoding="latin1", nrows=5)
except:
    try:
        df_crime_preview = pd.read_csv(arquivo_crime, sep=";", encoding="utf-8", nrows=5)
    except:
        df_crime_preview = pd.read_csv(arquivo_crime, sep=",", encoding="latin1", nrows=5)

df_crime_preview.head()

In [ ]:
#Ver nomes das colunas
print("Colunas encontradas:")
print(df_crime_preview.columns.tolist())

In [ ]:
#Mostrar todas as colunas numeradas
for i, col in enumerate(df_crime_preview.columns):
    print(f"{i+1}: {repr(col)}")

In [ ]:
#Limpar nomes de colunas
df_crime_preview.columns = df_crime_preview.columns.str.strip()

print("Colunas após limpeza:")
print(df_crime_preview.columns.tolist())

In [ ]:
#Detectar automaticamente a coluna de município
col_municipio = None

for col in df_crime_preview.columns:
    if "municip" in col.lower():
        col_municipio = col
        break

print("Coluna de município detectada:", col_municipio)

In [ ]:
#Carregar só a coluna de município corretamente
if col_municipio is not None:
    df_municipios = pd.read_csv(
        arquivo_crime,
        sep=";",
        encoding="latin1",
        usecols=[col_municipio]
    )
    df_municipios.columns = df_municipios.columns.str.strip()
    df_municipios.head()
else:
    print("Coluna de município não encontrada.")

In [ ]:
#Verificar se Passo Fundo está na base
if col_municipio is not None:
    municipios_unicos = (
        df_municipios[col_municipio]
        .dropna()
        .astype(str)
        .str.strip()
        .str.upper()
        .unique()
    )

    print("PASSO FUNDO" in municipios_unicos)
else:
    print("Não foi possível verificar os municípios.")

In [ ]:
#Ver os municípios mais frequentes
if col_municipio is not None:
    print(df_municipios[col_municipio].value_counts().head(20))
else:
    print("Não foi possível listar municípios.")

In [ ]:
#Inspeção inicial do arquivo meteorológico principal
arquivo_meteo = os.path.join(meteo_path, "PASSO_FUNDO_2021-01-01_2025-09-05.csv")

df_meteo_preview = pd.read_csv(arquivo_meteo, sep=None, engine='python', nrows=5)
df_meteo_preview

In [ ]:
#Ver colunas da meteorologia
for i, col in enumerate(df_meteo_preview.columns):
    print(f"{i+1}: {col}")

## Checkpoint da Etapa 1

Nesta etapa, foram realizadas as seguintes ações:

- Organização da estrutura de pastas do projeto no Google Drive
- Separação dos dados em **criminalidade** e **meteorologia**
- Identificação do arquivo principal de ocorrências criminais:
  - `criminalidade_ocorrencias_rs_2026.csv`
- Identificação do arquivo meteorológico principal para o município de Passo Fundo:
  - `PASSO_FUNDO_2021-01-01_2025-09-05.csv`
- Criação de um inventário dos arquivos disponíveis
- Inspeção inicial da estrutura das bases para preparação das próximas etapas

Com isso, os dados estão organizados e prontos para as etapas de concatenação, limpeza, padronização e integração.

Etapa 2 – Concatenação dos Dados de Criminalidade

In [20]:
# Importação dos dados de criminalidade de 2021 a 2026

df2026 = pd.read_csv("/content/drive/MyDrive/projeto_criminalidade_passo_fundo/data/raw/data_criminalidade/criminalidade_ocorrencias_rs_2026.csv", sep=";", encoding='latin-1')
df2025 = pd.read_csv("/content/drive/MyDrive/projeto_criminalidade_passo_fundo/data/raw/data_criminalidade/criminalidade_ocorrencias_rs_2025.csv", sep=";", encoding='latin-1')
df2024 = pd.read_csv("/content/drive/MyDrive/projeto_criminalidade_passo_fundo/data/raw/data_criminalidade/criminalidade_ocorrencias_rs_2024.csv", sep=";", encoding='latin-1')
df2023 = pd.read_csv("/content/drive/MyDrive/projeto_criminalidade_passo_fundo/data/raw/data_criminalidade/criminalidade_ocorrencias_rs_2023.csv", sep=";", encoding='latin-1')
df2022 = pd.read_csv("/content/drive/MyDrive/projeto_criminalidade_passo_fundo/data/raw/data_criminalidade/criminalidade_ocorrencias_rs_2022.csv", sep=";", encoding='latin-1')
df2021 = pd.read_csv("/content/drive/MyDrive/projeto_criminalidade_passo_fundo/data/raw/data_criminalidade/criminalidade_ocorrencias_rs_2021.csv", sep=";", encoding='latin-1')

/tmp/ipykernel_56849/1815937886.py:3: DtypeWarning: Columns (29,30,32,33,35,36,38,39,41,42,44,45,47,48,50,51,53,54,56,57,59,60,62,63,65,66,68,69,71,72,74,75,77,78,80,81,83,84) have mixed types. Specify dtype option on import or set low_memory=False.
  df2026 = pd.read_csv("/content/drive/MyDrive/projeto_criminalidade_passo_fundo/data/raw/data_criminalidade/criminalidade_ocorrencias_rs_2026.csv", sep=";", encoding='latin-1')
/tmp/ipykernel_56849/1815937886.py:4: DtypeWarning: Columns (23,24,26,27,29,30,32,33,35,36,38,39,41,42,44,45,47,48,50,51,53,54,56,57,59,60,62,63,65,66,68,69,71,72,74,75,77,78,80,81,83,84,86,87,89,90,92,93,95,96,98,99,101,102,104,105,107,108,110,111,113,114,116,117,119,120,122,123,125,126,128,129,131,132,134,135,137,138,140,141,143,144,146,147,149,150,152,153,155,156,158,159,161,162,164,165,167,168,170,171,173,174,176,177,179,180) have mixed types. Specify dtype option on import or set low_memory=False.
  df2025 = pd.read_csv("/content/drive/MyDrive/projeto_criminali

In [21]:
# Limpeza do df2026

df2026.drop(columns=[f"Unnamed: {x}" for x in range (14, 85)], inplace=True)
df2026.drop(columns="...", inplace=True)

df2026.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121882 entries, 0 to 121881
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Sequência           121882 non-null  int64  
 1   Data Fato           121882 non-null  object 
 2   Hora Fato           121882 non-null  object 
 3   Grupo Fato          121882 non-null  object 
 4   Tipo Enquadramento  121882 non-null  object 
 5   Tipo Fato           121882 non-null  object 
 6   Municipio Fato      121882 non-null  object 
 7   Local Fato          121882 non-null  object 
 8   Bairro              98625 non-null   object 
 9   Quantidade Vítimas  121882 non-null  int64  
 10  Idade Vítima        100212 non-null  float64
 11  Sexo Vítima         100165 non-null  object 
 12  Cor Vítima          100253 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 12.1+ MB


In [22]:
# Limpeza do df2025

df2025.drop(columns=[f"Unnamed: {x}" for x in range (14, 181)], inplace=True)
df2025.drop(columns="...", inplace=True)

df2025.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 763795 entries, 0 to 763794
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Sequência           763795 non-null  int64  
 1   Data Fato           763795 non-null  object 
 2   Hora Fato           763795 non-null  object 
 3   Grupo Fato          763795 non-null  object 
 4   Tipo Enquadramento  763795 non-null  object 
 5   Tipo Fato           763795 non-null  object 
 6   Municipio Fato      763795 non-null  object 
 7   Local Fato          763795 non-null  object 
 8   Bairro              616316 non-null  object 
 9   Quantidade Vítimas  763790 non-null  float64
 10  Idade Vítima        619363 non-null  float64
 11  Sexo Vítima         618493 non-null  object 
 12  Cor Vítima          619364 non-null  object 
dtypes: float64(2), int64(1), object(10)
memory usage: 75.8+ MB


In [23]:
# Limpeza do df2024

df2024.drop(columns=[f"Unnamed: {x}" for x in range (14, 127)], inplace=True)
df2024.drop(columns="...", inplace=True)

df2024.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 759848 entries, 0 to 759847
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Sequência           759848 non-null  int64  
 1   Data Fato           759848 non-null  object 
 2   Hora Fato           759848 non-null  object 
 3   Grupo Fato          759848 non-null  object 
 4   Tipo Enquadramento  759848 non-null  object 
 5   Tipo Fato           759848 non-null  object 
 6   Municipio Fato      759848 non-null  object 
 7   Local Fato          759848 non-null  object 
 8   Bairro              602185 non-null  object 
 9   Quantidade Vítimas  759843 non-null  float64
 10  Idade Vítima        615077 non-null  float64
 11  Sexo Vítima         614214 non-null  object 
 12  Cor Vítima          615078 non-null  object 
dtypes: float64(2), int64(1), object(10)
memory usage: 75.4+ MB


In [24]:
# Limpeza do df2023

df2023.drop(columns=[f"Unnamed: {x}" for x in range (14, 124)], inplace=True)
df2023.drop(columns="...", inplace=True)

df2023.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 806945 entries, 0 to 806944
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Sequência           806945 non-null  int64  
 1   Data Fato           806945 non-null  object 
 2   Hora Fato           806945 non-null  object 
 3   Grupo Fato          806945 non-null  object 
 4   Tipo Enquadramento  806945 non-null  object 
 5   Tipo Fato           806945 non-null  object 
 6   Municipio Fato      806945 non-null  object 
 7   Local Fato          806945 non-null  object 
 8   Bairro              637084 non-null  object 
 9   Quantidade Vítimas  806940 non-null  float64
 10  Idade Vítima        638156 non-null  float64
 11  Sexo Vítima         637676 non-null  object 
 12  Cor Vítima          638513 non-null  object 
dtypes: float64(2), int64(1), object(10)
memory usage: 80.0+ MB


In [25]:
# Limpeza do df2022

df2022.drop(columns=[f"Unnamed: {x}" for x in range (14, 118)], inplace=True)
df2022.drop(columns="...", inplace=True)

df2022.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 599648 entries, 0 to 599647
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Sequência           599648 non-null  object 
 1   Data Fato           599648 non-null  object 
 2   Hora Fato           599648 non-null  object 
 3   Grupo Fato          599648 non-null  object 
 4   Tipo Enquadramento  599648 non-null  object 
 5   Tipo Fato           599647 non-null  object 
 6   Municipio Fato      599647 non-null  object 
 7   Local Fato          599647 non-null  object 
 8   Bairro              492558 non-null  object 
 9   Quantidade Vítimas  599646 non-null  float64
 10  Idade Vítima        595635 non-null  float64
 11  Sexo Vítima         595971 non-null  object 
 12  Cor Vítima          596039 non-null  object 
dtypes: float64(2), object(11)
memory usage: 59.5+ MB


In [26]:
# Limpeza do df2021

df2021.drop(columns=[f"Unnamed: {x}" for x in range (14, 136)], inplace=True)
df2021.drop(columns="...", inplace=True)

df2021.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166150 entries, 0 to 166149
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Sequência           166150 non-null  int64  
 1   Data Fato           166150 non-null  object 
 2   Hora Fato           166150 non-null  object 
 3   Grupo Fato          166150 non-null  object 
 4   Tipo Enquadramento  166150 non-null  object 
 5   Tipo Fato           166150 non-null  object 
 6   Municipio Fato      166150 non-null  object 
 7   Local Fato          166150 non-null  object 
 8   Bairro              132645 non-null  object 
 9   Quantidade Vítimas  166150 non-null  int64  
 10  Idade Vítima        132008 non-null  float64
 11  Sexo Vítima         132010 non-null  object 
 12  Cor Vítima          132010 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 16.5+ MB


In [27]:
# Muda nomes das colunas

for dataframe in ([df2021, df2022, df2023, df2024, df2025, df2026]):
  dataframe.rename(columns={
  "Sequência": "sequencia",
  "Data Fato": "data_fato",
  "Hora Fato": "hora_fato",
  "Grupo Fato": "grupo_fato",
  "Tipo Enquadramento": "tipo_enquadramento",
  "Tipo Fato": "tipo_fato",
  "Municipio Fato": "municipio_fato",
  "Local Fato": "local_fato",
  "Bairro": "bairro",
  "Quantidade Vítimas": "quantidade_vitimas",
  "Idade Vítima": "idade_vitima",
  "Sexo Vítima": "sexo_vitima",
  "Cor Vítima": "cor_vitima"
}, inplace=True)

In [28]:
# Concatenação dos dataframes e mudança para tipos de dados coerentes

df = pd.concat([df2021, df2022, df2023, df2024, df2025, df2026], ignore_index=True)

df.grupo_fato = df.grupo_fato.astype("category")
df.tipo_enquadramento = df.tipo_enquadramento.astype("category")
df.tipo_fato = df.tipo_fato.astype("category")
df.municipio_fato = df.municipio_fato.astype("category")
df.local_fato = df.local_fato.astype("category")
df.sexo_vitima = df.sexo_vitima.astype("category")
df.cor_vitima = df.cor_vitima.astype("category")

df.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3218268 entries, 0 to 3218267
Data columns (total 13 columns):
 #   Column              Dtype   
---  ------              -----   
 0   sequencia           object  
 1   data_fato           object  
 2   hora_fato           object  
 3   grupo_fato          category
 4   tipo_enquadramento  category
 5   tipo_fato           category
 6   municipio_fato      category
 7   local_fato          category
 8   bairro              object  
 9   quantidade_vitimas  float64 
 10  idade_vitima        float64 
 11  sexo_vitima         category
 12  cor_vitima          category
dtypes: category(7), float64(2), object(4)
memory usage: 175.0+ MB


In [29]:
df.columns

Index(['sequencia', 'data_fato', 'hora_fato', 'grupo_fato',
       'tipo_enquadramento', 'tipo_fato', 'municipio_fato', 'local_fato',
       'bairro', 'quantidade_vitimas', 'idade_vitima', 'sexo_vitima',
       'cor_vitima'],
      dtype='object')

Etapa 3 – Limpeza e Padronização

In [30]:
import unicodedata

def padronizar(texto):
    texto = str(texto).lower().strip()
    # Remove acentos (ex: 'Homicídio' -> 'homicidio')
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return texto


In [31]:
# 1. Remover duplicatas
df = df.drop_duplicates()

# 2. Tratar valores ausentes (colunas completamente vazias)
df = df.dropna(axis=1, how='all')

# 3. Ajustar o formato de datas (YYYY-MM-DD)
df['data_fato'] = pd.to_datetime(df['data_fato'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['data_fato'])
df['data_fato'] = df['data_fato'].dt.strftime('%Y-%m-%d')

# 4. Corrigir inconsistências em colunas de texto
# (ex: 'Homicídio' vs 'homicidio' -> tudo vira 'homicidio')
colunas_texto = ['grupo_fato', 'tipo_enquadramento', 'tipo_fato',
                 'local_fato', 'sexo_vitima', 'cor_vitima']

for col in colunas_texto:
    mapa = {val: padronizar(val) for val in df[col].unique()}
    df[col] = df[col].map(mapa)

# Município em MAIÚSCULO para o merge da Etapa 4
df['municipio_fato'] = df['municipio_fato'].str.upper()

df.info(verbose=True)


<class 'pandas.core.frame.DataFrame'>
Index: 3218267 entries, 0 to 3218267
Data columns (total 13 columns):
 #   Column              Dtype   
---  ------              -----   
 0   sequencia           object  
 1   data_fato           object  
 2   hora_fato           object  
 3   grupo_fato          object  
 4   tipo_enquadramento  object  
 5   tipo_fato           category
 6   municipio_fato      object  
 7   local_fato          category
 8   bairro              object  
 9   quantidade_vitimas  float64 
 10  idade_vitima        float64 
 11  sexo_vitima         object  
 12  cor_vitima          object  
dtypes: category(2), float64(2), object(9)
memory usage: 300.8+ MB


#Etapa 4 – Integração com Dados Meteorológicos

## 1 - Importar o dataset meteorológico

In [32]:
# Função para remover linhas da estação antes do csv e retorna dados em df
import pandas as pd
from pathlib import Path

def pre_processar_estacoes(folder_path:str):
    list_data = []
    for file in Path(folder_path).iterdir():
        if file.is_file():
            with open(file, 'r', encoding='utf-8') as f:
                station_dict = {}
                for line in f:
                    if line.strip() == '':
                        break
                    key, value = line.strip().split(': ', 1)
                    station_dict[key.strip()] = value.strip()
            list_data.append(station_dict)
    df = pd.DataFrame(list_data)
    return df

In [33]:
# monta df com informações das estações
df_estacoes = pre_processar_estacoes(meteo_path)

df_estacoes.head()

,Nome,Codigo Estacao,Latitude,Longitude,Altitude,Situacao,Data Inicial,Data Final,Periodicidade da Medicao
0,BENTO GONCALVES,A840,-29.164581,-51.534202,623.27,Operante,2021-01-01,2025-09-05,Diaria
1,BAGE,A827,-31.34777777,-54.01333333,226.19,Operante,2021-01-01,2025-09-05,Diaria
2,CACAPAVA DO SUL,A812,-30.54527777,-53.46694443,420.82,Operante,2021-01-01,2025-09-05,Diaria
3,CAMBARA DO SUL,A897,-29.04916666,-50.14972221,1017,Operante,2021-01-01,2025-09-05,Diaria
4,CAMAQUA,A838,-30.80805555,-51.83416666,92.3,Operante,2021-01-01,2025-09-05,Diaria


In [34]:
# Função para montar df com dados meteorologicos por estação
def processar_dados_met(folder_path: str):
    df = pd.DataFrame()
    for file in Path(folder_path).iterdir():
        if file.is_file():
            new_df = pd.read_csv(file, sep=';', skiprows=10)
            new_df.drop('Unnamed: 6', axis=1, inplace=True)
            with open(file=file, mode='r') as f:
                key, value = f.readline().strip().split(':')
            new_df[key.strip()] = value.strip()
            df = pd.concat([df, new_df], ignore_index=True)
    return df

In [35]:
# Une dados de todos os arquivos em um só df
df_dados_met = processar_dados_met(meteo_path)
df_dados_met.head()

,Data Medicao,"PRECIPITACAO TOTAL, DIARIO (AUT)(mm)","TEMPERATURA MAXIMA, DIARIA (AUT)(°C)","TEMPERATURA MINIMA, DIARIA (AUT)(°C)","UMIDADE RELATIVA DO AR, MEDIA DIARIA (AUT)(%)","VENTO, VELOCIDADE MEDIA DIARIA (AUT)(m/s)",Nome
0,2021-01-01,0,"26,6","14,8","57,6","2,9",BENTO GONCALVES
1,2021-01-02,0,"27,9","15,2","52,9","3,5",BENTO GONCALVES
2,2021-01-03,0,"28,9","15,9","65,8","3,1",BENTO GONCALVES
3,2021-01-04,0,"30,4","17,4","58,7","3,8",BENTO GONCALVES
4,2021-01-05,0,"25,2","20,9","78,3","3,2",BENTO GONCALVES


In [36]:
# Reonmeia colunas para padrão mais programático
df_dados_met.rename(columns={
    'Nome': 'estacao',
    'Data Medicao': 'data',
    'PRECIPITACAO TOTAL, DIARIO (AUT)(mm)': 'precipitacao',
    'TEMPERATURA MAXIMA, DIARIA (AUT)(°C)': 'temperatura_max',
    'TEMPERATURA MINIMA, DIARIA (AUT)(°C)': 'temperatura_min',
    'UMIDADE RELATIVA DO AR, MEDIA DIARIA (AUT)(%)': 'umidade_relativa',
    'VENTO, VELOCIDADE MEDIA DIARIA (AUT)(m/s)': 'vel_vento_med',
}, inplace=True)
df_dados_met.head()

,data,precipitacao,temperatura_max,temperatura_min,umidade_relativa,vel_vento_med,estacao
0,2021-01-01,0,"26,6","14,8","57,6","2,9",BENTO GONCALVES
1,2021-01-02,0,"27,9","15,2","52,9","3,5",BENTO GONCALVES
2,2021-01-03,0,"28,9","15,9","65,8","3,1",BENTO GONCALVES
3,2021-01-04,0,"30,4","17,4","58,7","3,8",BENTO GONCALVES
4,2021-01-05,0,"25,2","20,9","78,3","3,2",BENTO GONCALVES


## 2 - Garantir que a coluna de data esteja no mesmo formato que o dataset de criminalidade.

In [37]:
# Formata data para padrão DD/MM/YYYY
from datetime import datetime
format_str = '%Y-%m-%d'
df_dados_met['data'] = df_dados_met['data'].map(
    lambda data: datetime.strftime(
        datetime.fromisoformat(data),
        format_str
    )
)
df_dados_met.head()

,data,precipitacao,temperatura_max,temperatura_min,umidade_relativa,vel_vento_med,estacao
0,2021-01-01,0,"26,6","14,8","57,6","2,9",BENTO GONCALVES
1,2021-01-02,0,"27,9","15,2","52,9","3,5",BENTO GONCALVES
2,2021-01-03,0,"28,9","15,9","65,8","3,1",BENTO GONCALVES
3,2021-01-04,0,"30,4","17,4","58,7","3,8",BENTO GONCALVES
4,2021-01-05,0,"25,2","20,9","78,3","3,2",BENTO GONCALVES


In [38]:
# Função para converter valores em float ao invés de string
def convert_string_to_float(line: pd.Series):
    new_line = []
    for value in line:
        if value == None:
            new_line.append(None)
        else:
            new_line.append(
                float(
                    str(value).replace(',', '.')
                )
            )
    return new_line

In [39]:
# Formata numeros para float ao invés de str
cols = [
    'precipitacao',
    'temperatura_max',
    'temperatura_min',
    'umidade_relativa',
    'vel_vento_med',
]
df_dados_met[cols] = df_dados_met[cols].apply(convert_string_to_float)
df_dados_met.dtypes

,0
data,object
precipitacao,float64
temperatura_max,float64
temperatura_min,float64
umidade_relativa,float64
vel_vento_med,float64
estacao,object


## 3 - Fazer o merge dos datasets (chave: data + cidade = Passo Fundo).

In [40]:
# Renomeia colunas dos dados meteorologicos para fazer o merge
df_dados_met.rename(columns={
    'data': 'data_fato',
    'estacao': 'municipio_fato',
}, inplace=True)
df_dados_met.columns

Index(['data_fato', 'precipitacao', 'temperatura_max', 'temperatura_min',
       'umidade_relativa', 'vel_vento_med', 'municipio_fato'],
      dtype='object')

In [41]:
# Faz o merge dos datasets
df_unified = pd.merge(
    left=df,
    right=df_dados_met,
    how='inner',
    on=['data_fato','municipio_fato'],
)
# df_unified.set_index('sequencia', inplace=True)
df_unified.isna().sum()

,0
sequencia,0
data_fato,0
hora_fato,0
grupo_fato,0
tipo_enquadramento,0
tipo_fato,0
municipio_fato,0
local_fato,0
bairro,146303
quantidade_vitimas,1


## 4 - Garantir que nenhuma coluna gerou dados vazios. Caso alguma informação esteja vazia, preencher com alguma informação consistente ou remover.

In [42]:
# Função para preencher informações numéricas vazias com média
def fillna_com_mean_varias_colunas(df: pd.DataFrame, columns: list):
    for column in columns:
        df[column].fillna(round(df[column].mean(), 2), inplace=True)
    return df

df_unified = fillna_com_mean_varias_colunas(df_unified, ['idade_vitima', 'precipitacao', 'temperatura_max', 'temperatura_min', 'umidade_relativa', 'vel_vento_med'])
df_unified.isna().sum()

/tmp/ipykernel_56849/256833448.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[column].fillna(round(df[column].mean(), 2), inplace=True)


,0
sequencia,0
data_fato,0
hora_fato,0
grupo_fato,0
tipo_enquadramento,0
tipo_fato,0
municipio_fato,0
local_fato,0
bairro,146303
quantidade_vitimas,1


In [43]:
# Remove demais linhas vaizas
df_unified.dropna(inplace=True)
df_unified.isna().sum()

,0
sequencia,0
data_fato,0
hora_fato,0
grupo_fato,0
tipo_enquadramento,0
tipo_fato,0
municipio_fato,0
local_fato,0
bairro,0
quantidade_vitimas,0


Etapa 5 – Transformações

In [44]:
sem_outliers = df_unified.copy()

q1 = sem_outliers['idade_vitima'].quantile(0.25)
q3 = sem_outliers['idade_vitima'].quantile(0.75)
iqr = q3 - q1

limite_superior = q3 + 1.5 * iqr
limite_inferior = q1 - 1.5 * iqr

print(limite_superior, limite_inferior)

filtro_inferior = sem_outliers['idade_vitima'] > limite_inferior
filtro_superior = sem_outliers['idade_vitima'] < limite_superior

sem_outliers = sem_outliers[(filtro_inferior) & (filtro_superior)]

77.5 1.5


In [45]:
q1 = sem_outliers['temperatura_max'].quantile(0.25)
q3 = sem_outliers['temperatura_max'].quantile(0.75)
iqr = q3 - q1

limite_superior = q3 + 1.5 * iqr
limite_inferior = q1 - 1.5 * iqr

print(limite_superior, limite_inferior)

filtro_inferior = sem_outliers['temperatura_max'] > limite_inferior
filtro_superior = sem_outliers['temperatura_max'] < limite_superior

sem_outliers = sem_outliers[(filtro_inferior) & (filtro_superior)]

41.2 9.2


In [46]:
q1 = sem_outliers['temperatura_min'].quantile(0.25)
q3 = sem_outliers['temperatura_min'].quantile(0.75)
iqr = q3 - q1

limite_superior = q3 + 1.5 * iqr
limite_inferior = q1 - 1.5 * iqr

print(limite_superior, limite_inferior)

filtro_inferior = sem_outliers['temperatura_min'] > limite_inferior
filtro_superior = sem_outliers['temperatura_min'] < limite_superior

sem_outliers = sem_outliers[(filtro_inferior) & (filtro_superior)]

28.05 2.0500000000000007


In [47]:
q1 = sem_outliers['precipitacao'].quantile(0.25)
q3 = sem_outliers['precipitacao'].quantile(0.75)
iqr = q3 - q1

limite_superior = q3 + 1.5 * iqr
limite_inferior = q1 - 1.5 * iqr

print(limite_superior, limite_inferior)

filtro_inferior = sem_outliers['precipitacao'] > limite_inferior
filtro_superior = sem_outliers['precipitacao'] < limite_superior

sem_outliers = sem_outliers[(filtro_inferior) & (filtro_superior)]

11.525000000000002 -6.915000000000001


In [48]:
q1 = sem_outliers['umidade_relativa'].quantile(0.25)
q3 = sem_outliers['umidade_relativa'].quantile(0.75)
iqr = q3 - q1

limite_superior = q3 + 1.5 * iqr
limite_inferior = q1 - 1.5 * iqr

print(limite_superior, limite_inferior)

filtro_inferior = sem_outliers['umidade_relativa'] > limite_inferior
filtro_superior = sem_outliers['umidade_relativa'] < limite_superior

sem_outliers = sem_outliers[(filtro_inferior) & (filtro_superior)]

103.10000000000002 48.69999999999998


In [49]:
q1 = sem_outliers['vel_vento_med'].quantile(0.25)
q3 = sem_outliers['vel_vento_med'].quantile(0.75)
iqr = q3 - q1

limite_superior = q3 + 1.5 * iqr
limite_inferior = q1 - 1.5 * iqr

print(limite_superior, limite_inferior)

filtro_inferior = sem_outliers['vel_vento_med'] > limite_inferior
filtro_superior = sem_outliers['vel_vento_med'] < limite_superior

sem_outliers = sem_outliers[(filtro_inferior) & (filtro_superior)]

5.050000000000001 -0.15000000000000013


In [50]:
colunas_numericas = [
    'idade_vitima',
    'quantidade_vitimas',
    'precipitacao',
    'temperatura_max',
    'temperatura_min',
    'umidade_relativa',
    'vel_vento_med'
]

for col in colunas_numericas:
    media = sem_outliers[col].mean()
    desvio = sem_outliers[col].std()

    sem_outliers[f'z_score_{col}'] = (
        (sem_outliers[col] - media) / desvio
    )

    min_val = sem_outliers[col].min()
    max_val = sem_outliers[col].max()

    sem_outliers[f'minmax_{col}'] = (
        (sem_outliers[col] - min_val) / (max_val - min_val)
    )

In [51]:
sem_outliers.head()

,sequencia,data_fato,hora_fato,grupo_fato,tipo_enquadramento,tipo_fato,municipio_fato,local_fato,bairro,quantidade_vitimas,...,z_score_precipitacao,minmax_precipitacao,z_score_temperatura_max,minmax_temperatura_max,z_score_temperatura_min,minmax_temperatura_min,z_score_umidade_relativa,minmax_umidade_relativa,z_score_vel_vento_med,minmax_vel_vento_med
0,3,2021-10-01,00:01:00,crimes,furto simples em residencia,consumado,SANTA ROSA,residencia,Cruzeiro,1.0,...,1.367247,0.404386,-0.238903,0.468553,1.067366,0.688716,1.270480,0.781676,0.101024,0.506
2,7,2021-10-01,00:01:00,crimes,furto abigeato,consumado,PASSO FUNDO,outros,ROD TRANSBRASILIANA,1.0,...,3.855366,0.929825,-0.453563,0.430818,0.193984,0.525292,1.549509,0.836257,0.675655,0.620
6,28,2021-10-01,00:10:00,crimes,lesao corporal,consumado,ERECHIM,via publica,José Bonifácio,1.0,...,1.367247,0.404386,-0.074331,0.497484,-0.011885,0.486770,0.070652,0.546979,0.101024,0.506
11,43,2021-10-01,00:30:00,crimes,furto qualificado,consumado,SANTA ROSA,residencia,Timbaúva,1.0,...,1.367247,0.404386,-0.238903,0.468553,1.067366,0.688716,1.270480,0.781676,0.101024,0.506
16,65,2021-10-01,01:10:00,crimes,entorpecentes - trafico,consumado,PASSO FUNDO,residencia,Vila Popular,0.0,...,3.855366,0.929825,-0.453563,0.430818,0.193984,0.525292,1.549509,0.836257,0.675655,0.620


In [52]:
sem_outliers['hora_fato'] = pd.to_datetime(
    sem_outliers['hora_fato'],
    errors='coerce'
)

/tmp/ipykernel_56849/2778358503.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sem_outliers['hora_fato'] = pd.to_datetime(


In [53]:
def classificar_periodo(hora):
    if pd.isna(hora):
        return None

    h = hora.hour

    if 0 <= h <= 5:
        return 1  # madrugada
    elif 6 <= h <= 11:
        return 2  # manhã
    elif 12 <= h <= 17:
        return 3  # tarde
    else:
        return 4  # noite

In [54]:
sem_outliers['periodo_dia'] = sem_outliers['hora_fato'].apply(classificar_periodo)

In [55]:
df


,sequencia,data_fato,hora_fato,grupo_fato,tipo_enquadramento,tipo_fato,municipio_fato,local_fato,bairro,quantidade_vitimas,idade_vitima,sexo_vitima,cor_vitima
0,1,2021-10-01,00:01:00,crimes,incendio doloso,consumado,NOVA ALVORADA,residencia,Jardim Bela Vista,1.0,20.0,masculino,branca
1,2,2021-10-01,00:01:00,crimes,violencia psicologica contra mulher art 147b,consumado,SANTA CRUZ DO SUL,residencia,Santa Vitória,1.0,66.0,feminino,branca
2,3,2021-10-01,00:01:00,crimes,furto simples em residencia,consumado,SANTA ROSA,residencia,Cruzeiro,1.0,49.0,feminino,branca
3,4,2021-10-01,00:01:00,crimes,importunacao sexual,consumado,RIO GRANDE,via publica,Vila Santa Tereza,1.0,42.0,feminino,branca
4,5,2021-10-01,00:01:00,crimes,outras fraudes,consumado,PORTO ALEGRE,via publica,Azenha,1.0,51.0,feminino,parda
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3218263,121878,2026-02-28,23:50:00,crimes,embriaguez - art. 306,consumado,CACHOEIRINHA,via publica,Vl P Pora,0.0,NaN,nan,nan
3218264,121879,2026-02-28,23:50:00,contravencoes,vias de fato,consumado,CAPAO DA CANOA,residencia,Arco Iris,1.0,42.0,masculino,branca
3218265,121880,2026-02-28,23:50:00,crimes,ameaca,consumado,GUAPORE,residencia,Planalto,1.0,38.0,feminino,preta
3218266,121881,2026-02-28,23:51:00,crimes,furto/arrombamento de residencia,consumado,SAPUCAIA DO SUL,residencia,Vargas,1.0,43.0,masculino,branca


In [56]:
sem_outliers['hora_fato'] = pd.to_datetime(sem_outliers['hora_fato'])

def periodo(hora):
    h = hora.hour
    if 0 <= h <= 5:
        return 1
    elif 6 <= h <= 11:
        return 2
    elif 12 <= h <= 17:
        return 3
    else:
        return 4

sem_outliers['periodo_dia'] = sem_outliers['hora_fato'].apply(periodo)

sem_outliers[['hora_fato','periodo_dia']].head(20)

,hora_fato,periodo_dia
0,2026-04-09 00:01:00,1
2,2026-04-09 00:01:00,1
6,2026-04-09 00:10:00,1
11,2026-04-09 00:30:00,1
16,2026-04-09 01:10:00,1
18,2026-04-09 01:19:00,1
24,2026-04-09 02:00:00,1
29,2026-04-09 02:10:00,1
31,2026-04-09 02:36:00,1
32,2026-04-09 03:00:00,1


In [57]:
sem_outliers

,sequencia,data_fato,hora_fato,grupo_fato,tipo_enquadramento,tipo_fato,municipio_fato,local_fato,bairro,quantidade_vitimas,...,minmax_precipitacao,z_score_temperatura_max,minmax_temperatura_max,z_score_temperatura_min,minmax_temperatura_min,z_score_umidade_relativa,minmax_umidade_relativa,z_score_vel_vento_med,minmax_vel_vento_med,periodo_dia
0,3,2021-10-01,2026-04-09 00:01:00,crimes,furto simples em residencia,consumado,SANTA ROSA,residencia,Cruzeiro,1.0,...,0.404386,-0.238903,0.468553,1.067366,0.688716,1.270480,0.781676,0.101024,0.506,1
2,7,2021-10-01,2026-04-09 00:01:00,crimes,furto abigeato,consumado,PASSO FUNDO,outros,ROD TRANSBRASILIANA,1.0,...,0.929825,-0.453563,0.430818,0.193984,0.525292,1.549509,0.836257,0.675655,0.620,1
6,28,2021-10-01,2026-04-09 00:10:00,crimes,lesao corporal,consumado,ERECHIM,via publica,José Bonifácio,1.0,...,0.404386,-0.074331,0.497484,-0.011885,0.486770,0.070652,0.546979,0.101024,0.506,1
11,43,2021-10-01,2026-04-09 00:30:00,crimes,furto qualificado,consumado,SANTA ROSA,residencia,Timbaúva,1.0,...,0.404386,-0.238903,0.468553,1.067366,0.688716,1.270480,0.781676,0.101024,0.506,1
16,65,2021-10-01,2026-04-09 01:10:00,crimes,entorpecentes - trafico,consumado,PASSO FUNDO,residencia,Vila Popular,0.0,...,0.929825,-0.453563,0.430818,0.193984,0.525292,1.549509,0.836257,0.675655,0.620,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635994,526379,2025-09-05,2026-04-09 23:00:00,crimes,violencia psicologica contra mulher art 147b,consumado,URUGUAIANA,outros,Santana,1.0,...,0.404386,-0.074331,0.497484,-0.011885,0.486770,0.070652,0.546979,0.101024,0.506,4
635995,526391,2025-09-05,2026-04-09 23:06:00,crimes,favorecimento real,consumado,PASSO FUNDO,outros,São Luiz Gonzaga,0.0,...,0.404386,-0.074331,0.497484,-0.011885,0.486770,0.070652,0.546979,0.101024,0.506,4
635999,526405,2025-09-05,2026-04-09 23:30:00,crimes,lesao corporal culposa direcao veic automotor,consumado,CAMAQUA,via publica,Centro,0.0,...,0.404386,-0.074331,0.497484,-0.011885,0.486770,0.070652,0.546979,0.101024,0.506,4
636000,526413,2025-09-05,2026-04-09 23:45:00,crimes,descumprimento de medida protetiva de urgencia,consumado,PASSO FUNDO,estabelecimento comercial,CENTRO,1.0,...,0.404386,-0.074331,0.497484,-0.011885,0.486770,0.070652,0.546979,0.101024,0.506,4


In [58]:
Etapa 6 – Exploração e Descoberta (Balões 6, 7, 8...)

SyntaxError: invalid character '–' (U+2013) (1460095424.py, line 1)

In [ ]:
Balão 7 – Correlações:

In [ ]:
Balão 8 – Insight relevante:

In [ ]:
Balão 9 – Storytelling: